# LLM Performance Comparison Notebook

In [ ]:
!pip install transformers accelerate torch sentencepiece --quiet

## Setup and Authentication

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Library Imports

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

### Llama 3.2 1B

## Model Loading and Verification

In [ ]:
model_id_llama = "meta-llama/Llama-3.2-1B-Instruct" # Define model ID

tokenizer_llama = AutoTokenizer.from_pretrained(model_id_llama) # Load tokenizer
model_llama = AutoModelForCausalLM.from_pretrained( # Load model
    model_id_llama,
    torch_dtype=torch.bfloat16,
    device_map="auto"
    )
print(f"Llama 3.2 1B loaded.")

In [ ]:
_prompt = "Hi! How are you today?" # Simple input prompt
_inputs = tokenizer_llama(_prompt, return_tensors="pt").to(model_llama.device) # Tokenize and move to device
_outputs = model_llama.generate(**_inputs, max_new_tokens=5) # Generate response

# Decode and print verification message
print(f"Llama Verification Ok: {tokenizer_llama.decode(_outputs[0])}")

del _prompt, _inputs, _outputs # Clean up temporary variables

### GPU Status

In [ ]:
!nvidia-smi

### Gemma 2 2B

In [ ]:
model_id_gemma = "google/gemma-2-2b-it" # Define model ID
tokenizer_gemma = AutoTokenizer.from_pretrained(model_id_gemma) # Load tokenizer
model_gemma = AutoModelForCausalLM.from_pretrained( # Load model
    model_id_gemma,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"Gemma 2 2B loaded.")

In [ ]:
print("Running quick verification for Gemma...")
_prompt = "Hi! How are you today?" # Simple input prompt
_inputs = tokenizer_gemma(_prompt, return_tensors="pt").to(model_gemma.device) # Tokenize and move to device
_outputs = model_gemma.generate(**_inputs, max_new_tokens=5) # Generate response

# Decode and print verification message
print(f"Gemma Verification Ok: {tokenizer_gemma.decode(_outputs[0])}")

del _prompt, _inputs, _outputs # Clean up temporary variables

### GPU Status

In [ ]:
!nvidia-smi

### Qwen 2.5 0.5B

In [ ]:
model_id_qwen = "Qwen/Qwen2.5-0.5B-Instruct" # Define model ID
tokenizer_qwen = AutoTokenizer.from_pretrained(model_id_qwen) # Load tokenizer
model_qwen = AutoModelForCausalLM.from_pretrained( # Load model
    model_id_qwen,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"Qwen 2.5 0.5B loaded.")

print("Running quick verification for Qwen...")
_prompt = "Hi! How are you today?" # Simple input prompt
_inputs = tokenizer_qwen(_prompt, return_tensors="pt").to(model_qwen.device) # Tokenize and move to device
_outputs = model_qwen.generate(**_inputs, max_new_tokens=5) # Generate response
print(f"Qwen Verification Ok: {tokenizer_qwen.decode(_outputs[0])}")

del _prompt, _inputs, _outputs # Clean up temporary variables

In [ ]:
!nvidia-smi

## Utility Functions for Text Generation

In [ ]:
import time

def generate_text(model, tokenizer, prompt, max_new_tokens):
  start_time = time.time() # Record start time

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # Tokenize and move to device
  outputs = model.generate(**inputs, max_new_tokens=max_new_tokens) # Generate text

  end_time = time.time() # Record end time

  # Process output tokens and text
  output_ids = outputs[0]
  input_token_len = inputs.input_ids.shape[1]
  generated_ids = output_ids[input_token_len:]
  generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
  num_generated_tokens = len(generated_ids)

  duration = end_time - start_time
  tokens_per_sec = num_generated_tokens / duration

  print(f"Generated {num_generated_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens/sec)")
  print(f"Output:\n {generated_text}")

  return generated_text, tokens_per_sec # Return generated text and speed

## Prompt 1: Climate Description

In [ ]:
prompt = "Describe the year-round climate in London, England"

### Generate Responses for Prompt 1

In [ ]:
# Generate text using Llama 3.2 1B with Prompt 1
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt, max_new_tokens=500)

In [ ]:
# Generate text using Gemma 2 2B with Prompt 1
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500)

In [ ]:
# Generate text using Qwen 2.5 0.5B with Prompt 1
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt, max_new_tokens=500)

## Prompt 2: Logic Puzzle

In [ ]:
prompt_2 = """Scenario:
Four colored cups – Red, Blue, Green, and Yellow – are arranged in a circle.
One cup has a hidden star.

Facts:

The cup with the star is not Red.
The cup with the star is directly next to the Blue cup.
The Green cup is directly between the Yellow cup and the Red cup (when going around the circle in one direction).
Question:
Which color cup has the star? Just give the correct answer"""

### Generate Responses for Prompt 2

In [ ]:
# Generate text using Llama 3.2 1B with Prompt 2
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_2, max_new_tokens=500)

In [ ]:
# Generate text using Gemma 2 2B with Prompt 2
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_2, max_new_tokens=500)

In [ ]:
# Generate text using Qwen 2.5 0.5B with Prompt 2
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt_2, max_new_tokens=500)

## Exploring Temperature Parameter

In [ ]:
prompt = "Write a short story about a time-traveling cat"

### Generation Function for Temperature Control

In [ ]:
def generate_text_with_temp(model, tokenizer, prompt, max_new_tokens, temperature):
    start_time = time.time() # Record start time

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # Tokenize and move to device
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens, # Max tokens to generate
        temperature=temperature,       # Sampling temperature
        do_sample=True,                # Enable sampling for temperature
        #pad_token_id=tokenizer.eos_token_id
    )
    end_time = time.time() # Record end time

    # Process output tokens and text
    output_ids = outputs[0]
    input_token_len = inputs.input_ids.shape[1]
    generated_ids = output_ids[input_token_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    num_generated_tokens = len(generated_ids)

    duration = end_time - start_time
    tokens_per_sec = num_generated_tokens / duration
    print(f"Generated {num_generated_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens/sec)")
    print(f"Output:\n {generated_text}")

    return generated_text, tokens_per_sec # Return generated text and speed

### Low Temperature Generation (0.2)

In [ ]:
# Generate with Gemma 2 2B at low temperature
_ = generate_text_with_temp(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500, temperature=0.2)

### High Temperature Generation (1.0)

In [ ]:
# Generate with Gemma 2 2B at high temperature
_ = generate_text_with_temp(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500, temperature=1.0)